# Setup:
1. Create environment:
    In terminal, run:
    
    `conda env create -n suite2p_preprocessing_env -f studio/app/optinist/wrappers/conda/expdb_batch.yaml`

    `conda activate suite2p_preprocessing_env`

2. Install additional packages:

   `pip install pynwb imageio ipython jupyter notebook plotly "pydantic<2.0.0" python-dotenv uvicorn xmltodict bcrypt`
  - If running in VS code, you may need to restart and/or select the correct environment with "Python: Select Interpreter"

3. Run this notebook

Note: This notebook demonstrates the ExpDB preprocessing workflow:
microscope_database -> preprocessing -> suite2p_preprocessing -> analyze_stats

In [ ]:
import os
import sys
import uuid
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('.'))

# Import OptiNiSt core data modules
from studio.app.dir_path import DIRPATH
from studio.app.common.dataclass import ImageData
from studio.app.optinist.dataclass import MicroscopeExpdbData
# Import preprocessing and analysis modules
from studio.app.optinist.wrappers.expdb.preprocessing import preprocessing
from studio.app.optinist.wrappers.suite2p.suite2p_preprocessing import suite2p_preprocessing
from studio.app.optinist.wrappers.expdb.analyze_stats import analyze_stats

import numpy as np
import pandas as pd

# Import visualization modules
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

# Create input directories based on default saving path
input_dir = os.path.join(DIRPATH.INPUT_DIR, "1") 
os.makedirs(input_dir, exist_ok=True)
unique_id = str(uuid.uuid4())[:8]  # Generate 8-char unique ID

In [ ]:
# Step 1: Load microscope data
# For this demo, we'll use a sample TIFF file
# In production, you would use ND2Reader for Nikon .nd2 files

input_file = os.path.join(input_dir, "sample_mouse2p_image.tiff")

# Create MicroscopeExpdbData wrapper
# Note: For .nd2 files, use: microscope_data = MicroscopeExpdbData(path_to_nd2_file)
microscope_data = MicroscopeExpdbData(input_file)

print(f"Loaded microscope data from: {input_file}")

In [ ]:
# Step 2: Set parameters for preprocessing
# This step performs phase correction and optional registration

preprocessing_params = {
    'preprocessing': {
        'first_dim': 0,          # First dimension for phase correction (0=Y, 1=X)
        'period': 10,            # Period for stack averaging
        'runs': 10,              # Number of runs for averaging
        'do_realign': True,      # Enable registration/realignment
        'usfac': 10,             # Upsampling factor for registration (2D)
        'le': 6,                 # Local extrema parameter (3D)
        'shift_method': 'cv2'    # Shift method for 3D: 'cv2' or 'scipy'
    }
}

In [ ]:
# Create output directory for preprocessing
preproc_function_id = f"preprocessing_{unique_id}"
preproc_output_dir = os.path.join(DIRPATH.OUTPUT_DIR, "1", unique_id, preproc_function_id)
os.makedirs(preproc_output_dir, exist_ok=True)

In [ ]:
# Run preprocessing
print("Running preprocessing (phase correction and registration)...")
ret_preproc = preprocessing(microscope_data, preproc_output_dir, preprocessing_params)
print(f"Preprocessing complete. Output keys: {list(ret_preproc.keys())}")

In [ ]:
# Step 3: Set parameters for suite2p_preprocessing
# This step performs ROI detection and creates ExpDB .mat files

suite2p_preprocessing_params = {
    'suite2p_preprocessing': {
        # Cell detection parameters
        'tau': 1.25,                    # Timescale of calcium indicator
        'fs': 30.0,                     # Sampling rate (Hz)
        'threshold_scaling': 1.0,       # Detection threshold multiplier
        'max_overlap': 0.75,            # Maximum ROI overlap
        'spatial_hp_detect': 25,        # Spatial high-pass filter window
        'connected': True,              # Use connected components
        'high_pass': 100,               # Temporal high-pass filter
        
        # ROI extraction parameters
        'neuropil_extract': True,       # Extract neuropil traces
        'inner_neuropil_radius': 2,     # Pixels between ROI and neuropil
        'min_neuropil_pixels': 350,     # Minimum neuropil pixels
        
        # ExpDB-specific parameters
        'neucoeff': 0.7,                # Neuropil contamination coefficient
        'normalize_by_energy': True,    # Apply energy normalization (recommended)
        
        # Visualization parameters
        'roi_thr_bool': False,          # Apply energy thresholding to ROI pixels
        'roi_thr': 0.9,                 # ROI pixel energy threshold
        
        # Output control
        'create_Yr': False,             # Create Yr.mat (large file, optional)
        'create_C_or': False,           # Create C_or.mat (optional)
        'validate_outputs': True,       # Validate .mat files
        'require_trialstructure': False # Require trial structure file (set False for testing)
    }
}

In [ ]:
# Create output directory for suite2p_preprocessing
s2p_preproc_function_id = f"suite2p_preprocessing_{unique_id}"
s2p_preproc_output_dir = os.path.join(DIRPATH.OUTPUT_DIR, "1", unique_id, s2p_preproc_function_id)
os.makedirs(s2p_preproc_output_dir, exist_ok=True)

In [ ]:
# Run suite2p_preprocessing
print("Running suite2p_preprocessing (ROI detection and .mat file creation)...")
ret_s2p_preproc = suite2p_preprocessing(
    ret_preproc['stack'], 
    s2p_preproc_output_dir, 
    suite2p_preprocessing_params
)
print(f"Suite2p preprocessing complete. Output keys: {list(ret_s2p_preproc.keys())}")

In [ ]:
# Step 4: Set parameters for analyze_stats
# This step performs statistical analysis for orientation/direction tuning

analyze_stats_params = {
    'stat_file_convert': {
        'normalize_TC': True,           # Normalize time courses
        'remove_blank': True,           # Remove blank trials
    },
    'anova1_mult': {
        'alpha': 0.05,                  # Significance level
    },
    'vector_average': {
        'use_circular': True,           # Use circular statistics
    },
    'curvefit_tuning': {
        'fit_method': 'vonmises',       # Tuning curve fit method
    }
}

In [ ]:
# Create output directory for analyze_stats
stats_function_id = f"analyze_stats_{unique_id}"
stats_output_dir = os.path.join(DIRPATH.OUTPUT_DIR, "1", unique_id, stats_function_id)
os.makedirs(stats_output_dir, exist_ok=True)

In [ ]:
# Run analyze_stats
# Note: This requires trial structure data to be present
print("Running analyze_stats (statistical analysis)...")
try:
    ret_stats = analyze_stats(
        ret_s2p_preproc['processed_data'], 
        stats_output_dir, 
        analyze_stats_params
    )
    print(f"Statistical analysis complete. Output keys: {list(ret_stats.keys())}")
except Exception as e:
    print(f"Error running analyze_stats: {e}")
    print("This is expected if trial structure file is not available.")
    ret_stats = None

In [ ]:
# Visualize Suite2p preprocessing results

# Get data from the output
mean_img = ret_s2p_preproc['mean_image'].data
max_proj = ret_s2p_preproc['max_proj'].data
Vcorr = ret_s2p_preproc['Vcorr'].data
cell_roi = ret_s2p_preproc['cell_roi'].data
F = ret_s2p_preproc['fluorescence'].data
iscell = ret_s2p_preproc['iscell'].data

# Create subplot figure
fig = make_subplots(
    rows=3, cols=2, 
    subplot_titles=(
        'Mean Image', 'Max Projection',
        'Correlation Image', 'Cell ROIs',
        'Mean Fluorescence', 'Individual Cell Traces'
    ),
    vertical_spacing=0.10,
    horizontal_spacing=0.12
)

# 1. Mean Image
fig.add_trace(
    go.Heatmap(
        z=mean_img, 
        colorscale='gray',
        showscale=False,
        name='Mean Image'
    ),
    row=1, col=1
)

# 2. Max Projection
fig.add_trace(
    go.Heatmap(
        z=max_proj,
        colorscale='gray',
        showscale=False,
        name='Max Projection'
    ),
    row=1, col=2
)

# 3. Correlation Image
fig.add_trace(
    go.Heatmap(
        z=Vcorr,
        colorscale='viridis',
        showscale=True,
        name='Correlation',
        colorbar=dict(
            title='Corr',
            len=0.25,
            y=0.5
        )
    ),
    row=2, col=1
)

# 4. Cell ROIs
fig.add_trace(
    go.Heatmap(
        z=cell_roi,
        colorscale='viridis',
        showscale=True,
        name='Cell ROIs',
        colorbar=dict(
            title='ROI #',
            len=0.25,
            y=0.5
        )
    ),
    row=2, col=2
)

# 5. Mean Fluorescence
mean_fluorescence = np.mean(F, axis=0)
time_points = np.arange(len(mean_fluorescence))

fig.add_trace(
    go.Scatter(
        x=time_points,
        y=mean_fluorescence,
        mode='lines',
        name='Mean Fluorescence',
        showlegend=False
    ),
    row=3, col=1
)

# 6. Individual Cell Traces
cell_indices = np.where(iscell == 1)[0]
n_cells_to_plot = min(10, len(cell_indices))
colors = px.colors.qualitative.Set3

for i in range(n_cells_to_plot):
    cell_idx = cell_indices[i]
    color = colors[i % len(colors)]
    fig.add_trace(
        go.Scatter(
            x=time_points,
            y=F[cell_idx, :],
            mode='lines',
            name=f'Cell {cell_idx+1}',
            line=dict(color=color, width=1),
            opacity=0.7,
            showlegend=(i < 5)  # Only show first 5 in legend
        ),
        row=3, col=2
    )

# Update layout
fig.update_layout(
    height=900,
    width=1200,
    title=dict(
        text=f"Suite2p Preprocessing Results ({np.sum(iscell)} cells detected)",
        x=0.5,
        y=0.98
    ),
    showlegend=True
)

# Update axes
for row in range(1, 4):
    for col in range(1, 3):
        if row <= 2:  # Images
            fig.update_xaxes(title_text="X", row=row, col=col)
            fig.update_yaxes(title_text="Y", row=row, col=col)
        else:  # Traces
            fig.update_xaxes(title_text="Time (frames)", row=row, col=col)
            fig.update_yaxes(title_text="Fluorescence", row=row, col=col)

fig.show()

In [ ]:
# Visualize analyze_stats results (if available)

if ret_stats is not None:
    # Get statistical analysis outputs
    stat_data = ret_stats['stat']
    
    # Create visualization of orientation/direction selectivity
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Preferred Direction',
            'Direction Selectivity',
            'Orientation Selectivity',
            'Tuning Curve Example'
        ),
        specs=[
            [{'type': 'scatter'}, {'type': 'histogram'}],
            [{'type': 'histogram'}, {'type': 'scatter'}]
        ],
        vertical_spacing=0.15,
        horizontal_spacing=0.12
    )
    
    # 1. Preferred Direction (polar-like scatter)
    pref_dir = ret_stats['preferred_direction'].data
    dir_sel = ret_stats['direction_selectivity'].data
    
    # Convert to radians for polar representation
    pref_dir_rad = np.deg2rad(pref_dir)
    x_vals = dir_sel * np.cos(pref_dir_rad)
    y_vals = dir_sel * np.sin(pref_dir_rad)
    
    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=y_vals,
            mode='markers',
            marker=dict(
                size=8,
                color=pref_dir,
                colorscale='hsv',
                showscale=True,
                colorbar=dict(
                    title='Direction (°)',
                    len=0.4,
                    y=0.8
                )
            ),
            text=[f"Cell {i+1}: {d:.1f}°" for i, d in enumerate(pref_dir)],
            hovertemplate='%{text}<br>DSI: %{marker.color:.2f}<extra></extra>',
            name='Cells'
        ),
        row=1, col=1
    )
    
    # 2. Direction Selectivity histogram
    fig.add_trace(
        go.Histogram(
            x=dir_sel,
            nbinsx=20,
            name='DSI',
            marker_color='steelblue'
        ),
        row=1, col=2
    )
    
    # 3. Orientation Selectivity histogram
    ori_sel = ret_stats['orientation_selectivity'].data
    fig.add_trace(
        go.Histogram(
            x=ori_sel,
            nbinsx=20,
            name='OSI',
            marker_color='coral'
        ),
        row=2, col=1
    )
    
    # 4. Example tuning curve (first selective cell)
    tuning_curves = ret_stats['tuning_curve'].data
    if len(tuning_curves) > 0:
        # Find most direction-selective cell
        best_cell_idx = np.argmax(dir_sel)
        tuning_curve = tuning_curves[best_cell_idx]
        
        # Assume evenly spaced directions
        n_dirs = len(tuning_curve)
        directions = np.linspace(0, 360, n_dirs, endpoint=False)
        
        fig.add_trace(
            go.Scatter(
                x=directions,
                y=tuning_curve,
                mode='lines+markers',
                name=f'Cell {best_cell_idx+1}',
                line=dict(color='darkgreen', width=2),
                marker=dict(size=8)
            ),
            row=2, col=2
        )
    
    # Update layout
    fig.update_layout(
        height=800,
        width=1200,
        title=dict(
            text="Orientation/Direction Selectivity Analysis",
            x=0.5,
            y=0.98
        ),
        showlegend=False
    )
    
    # Update axes
    fig.update_xaxes(title_text="X Position", row=1, col=1)
    fig.update_yaxes(title_text="Y Position", row=1, col=1)
    fig.update_xaxes(title_text="Direction Selectivity Index", row=1, col=2)
    fig.update_yaxes(title_text="Count", row=1, col=2)
    fig.update_xaxes(title_text="Orientation Selectivity Index", row=2, col=1)
    fig.update_yaxes(title_text="Count", row=2, col=1)
    fig.update_xaxes(title_text="Direction (°)", row=2, col=2)
    fig.update_yaxes(title_text="Response", row=2, col=2)
    
    fig.show()
    
    # Print summary statistics
    print("\n=== Statistical Analysis Summary ===")
    print(f"Total cells analyzed: {len(dir_sel)}")
    print(f"Mean Direction Selectivity Index: {np.mean(dir_sel):.3f} ± {np.std(dir_sel):.3f}")
    print(f"Mean Orientation Selectivity Index: {np.mean(ori_sel):.3f} ± {np.std(ori_sel):.3f}")
    print(f"Direction-selective cells (DSI > 0.3): {np.sum(dir_sel > 0.3)}")
    print(f"Orientation-selective cells (OSI > 0.3): {np.sum(ori_sel > 0.3)}")
else:
    print("Statistical analysis results not available (trial structure required)")
    print("To run the complete workflow, you need:")
    print("1. Trial structure .mat file in the ExpDB directory")
    print("2. Set require_trialstructure=True in suite2p_preprocessing_params")

In [ ]:
# Summary of the workflow

print("\n=== Workflow Summary ===")
print("This notebook demonstrates the complete ExpDB preprocessing pipeline:\n")
print("1. microscope_database (ND2Reader or TIFF)")
print("   └─> Loads microscope image data\n")
print("2. preprocessing")
print("   └─> Phase correction, averaging, and registration")
print(f"   └─> Output: {ret_preproc['stack'].data.shape} image stack\n")
print("3. suite2p_preprocessing")
print("   └─> Suite2p ROI detection")
print("   └─> Fluorescence extraction with neuropil correction")
print("   └─> Creates ExpDB-compatible .mat files (timecourse.mat)")
print(f"   └─> Detected: {np.sum(iscell)} cells\n")
if ret_stats is not None:
    print("4. analyze_stats")
    print("   └─> Statistical analysis (ANOVA, tuning curves)")
    print("   └─> Direction and orientation selectivity metrics")
    print(f"   └─> Analyzed: {len(ret_stats['stat'].data)} cells")
else:
    print("4. analyze_stats")
    print("   └─> Not run (requires trial structure file)")
    print("   └─> See cell above for requirements")
